# 01 数据准备

**目标**：从原始长文档构造 NIAH 测试数据集，输出 `data/processed/niah_dataset.jsonl`

**步骤**：
1. 准备原始文档（放入 `data/raw/`）
2. 预览文档内容
3. 配置评测参数
4. 生成数据集并验证

In [2]:
import sys
sys.path.insert(0, '..')  # 让 notebook 能找到 src/

import json
from pathlib import Path
import pandas as pd
import yaml

from src.data_prep import (
    load_document,
    build_niah_samples,
    save_jsonl,
    load_jsonl,
    NEEDLES_ZH,
)

# 加载配置
config = yaml.safe_load(open('../configs/eval_config.yaml', encoding='utf-8'))
print('配置加载成功')
print(f"测试上下文长度: {config['eval']['context_lengths']}")
print(f"Needle 深度点: {config['eval']['depth_percentages']}")

配置加载成功
测试上下文长度: [2000, 4000, 8000, 16000, 32000, 64000]
Needle 深度点: [0, 10, 25, 50, 75, 90, 100]


## Step 1：检查 data/raw/ 下的文档

⚠️ **如果 data/raw/ 为空**，请先放入中文长文档：
- 上市公司年报（从巨潮资讯 http://www.cninfo.com.cn 下载 PDF 转文本）
- 中文学术论文（找 > 2 万字的综述）
- 中文小说（古典文学、现代文学，Project Gutenberg 有部分）
- 建议每篇 **> 5 万字符**，总量 **> 20 万字符**

In [3]:
raw_dir = Path('../data/raw')
doc_files = sorted(raw_dir.glob('*.txt')) + sorted(raw_dir.glob('*.md'))

if not doc_files:
    print('⚠️  data/raw/ 下没有文档！请放入 .txt 或 .md 文件后重新运行此单元格。')
else:
    print(f'找到 {len(doc_files)} 篇文档：')
    for f in doc_files:
        content = load_document(str(f))
        print(f'  {f.name:40s}  {len(content):>8,} 字符')

找到 14 篇文档：
  三国.txt                                      24,012 字符
  中华人民共和国经济.txt                               19,489 字符
  中国历史.txt                                    55,476 字符
  人工智能.txt                                    15,191 字符
  佛教.txt                                      34,593 字符
  儒家.txt                                      23,876 字符
  唐朝.txt                                      35,215 字符
  宋朝.txt                                      42,373 字符
  数学.txt                                       7,615 字符
  汉朝.txt                                      38,561 字符
  物理学.txt                                     12,835 字符
  生物学.txt                                      5,553 字符
  量子力学.txt                                    32,293 字符
  README.md                                      572 字符


## Step 2：预览文档内容

In [4]:
if doc_files:
    doc = load_document(str(doc_files[0]))
    print('=== 文档前 500 字符 ===')
    print(doc[:500])
    print('\n=== 文档中间 500 字符 ===')
    mid = len(doc) // 2
    print(doc[mid:mid+500])
    print('\n=== 文档末尾 300 字符 ===')
    print(doc[-300:])

=== 文档前 500 字符 ===
# 三国

三國（狹義220年—280年，廣義184年／190年／208年—280年）是曹魏、蜀漢及孫吳三個國家並立的中國歷史時期，也是魏晉南北朝和六朝的開端時期。一般認為是從建安元年起算。東漢末年戰爭不斷，使得人口急劇下降，經濟嚴重損害，因此三國皆重視經濟發展，加上戰爭帶來的需求，各種技術都有許多進步。
東漢末年，漢廷因黃巾之亂、北宫伯玉之乱、黑山军起义、王芬谋废灵帝、张举张纯叛乱、外戚宦官火拼等一系列事件而动荡不安。184年漢靈帝時期，以張角三兄弟為首爆发黃巾之亂。為鎮壓黃巾，一方面放權到州牧、太守，一方面縱容地主組織私人武裝，對抗黃巾。董卓亂政並與關東諸勢力對抗後遷都長安，使得朝廷威信喪失，地方长官演变为独立军阀割據混戰。其中曹操擁護逃回洛陽的漢獻帝，遷都許昌。他擊敗多股勢力，最後在200年的官渡之戰擊敗北方最大勢力袁紹，大致掌控中國北方。曹操以優勢兵力南征荊州，但在208年冬天的赤壁之戰被孫權和劉備聯軍擊敗，形成三國鼎立的雛型。220年曹操病逝，其子曹丕接受漢獻帝禪讓而稱帝，國號「魏」，史稱曹魏，至此東漢滅亡，正式進入三國時期。
隔年以益州為主的劉備也以漢室宗親的身份稱帝，國號

=== 文档中间 500 字符 ===
，曹魏毌丘儉率軍擊敗高句麗。公孫康置帶方郡後與百濟聯姻，之後百濟併帶方郡而立國。日本的邪馬台國興起後，遣使納貢曹魏，魏明帝封邪馬台國女王卑彌呼為親魏倭王。魏晉以來天山以北及蒙古草原的民族主要有烏孫、堅昆、敕勒、丁零、呼揭、匈奴、鮮卑及烏桓等族。東漢之後匈奴分為南北；51年，南匈奴大多徙居在幷州中部的汾水一帶。188年，单于于扶罗趁中原內亂之际率軍入侵。202年，南匈奴歸附曹操後，曹操將南匈奴分成五部，每部立帥長，並派漢人監督。烏桓族長蹋頓與袁紹結盟，並獲得了單于的封號。205年，曹操擊敗袁尚，袁尚與袁熙兄弟逃至蹋頓處。而後曹操率精銳遠征烏桓，於白狼山斬殺蹋頓，降服烏桓。鮮卑在東漢末期由檀石槐統一，屢次入侵東漢，其死後鮮卑分裂為東部、中部及西部鮮卑。西部鮮卑軻比能重整鮮卑後兩度入侵曹魏，並響應諸葛亮攻魏。235年，曹魏幽州刺史王雄遣刺客將他暗殺，其勢瓦解。

西部方面，西羌於三國時期開始遷居中原，分佈於中國中部的山地地區。當時河西諸羌和武都、陰平的羌族分別歸附曹魏及蜀漢。這兩國相互攻伐時都徵召羌族參加作戰。

## Step 3：查看内置 Needle 列表

In [5]:
print(f'共 {len(NEEDLES_ZH)} 条内置 needle：\n')
for i, n in enumerate(NEEDLES_ZH):
    print(f'[{i+1}] Needle:   {n["needle"]}')
    print(f'    Question: {n["question"]}')
    print(f'    Answer:   {n["answer"]}\n')

共 8 条内置 needle：

[1] Needle:   张伟的私人银行账户密码是 X7K9-BLUE-2024。
    Question: 张伟的私人银行账户密码是什么？
    Answer:   X7K9-BLUE-2024

[2] Needle:   公司今年第三季度净利润为 47.3 亿元人民币。
    Question: 公司今年第三季度净利润是多少？
    Answer:   47.3亿元人民币

[3] Needle:   会议室的临时预订密码已更新为 Rainbow-512。
    Question: 会议室临时预订密码是什么？
    Answer:   Rainbow-512

[4] Needle:   内部项目代号为“暗星计划”，正式启动日期为 2024 年 11 月 8 日。
    Question: 内部项目代号是什么，启动日期是哪天？
    Answer:   暗星计划，2024年11月8日

[5] Needle:   李明的员工编号为 EMP-2024-08831。
    Question: 李明的员工编号是多少？
    Answer:   EMP-2024-08831

[6] Needle:   数据中心备用电源的启动口令是 GoldenKey-7749。
    Question: 数据中心备用电源的启动口令是什么？
    Answer:   GoldenKey-7749

[7] Needle:   本次合并的换股比例为每 3 股 A 公司股票换 1 股 B 公司股票。
    Question: 本次合并的换股比例是多少？
    Answer:   每3股A公司股票换1股B公司股票

[8] Needle:   首席技术官王磊的直线电话是 010-88887777 转 301。
    Question: 首席技术官王磊的直线电话是多少？
    Answer:   010-88887777转301



## Step 4：生成 NIAH 数据集

调整下方参数控制样本量和 API 费用：

In [ ]:
# ── 可调参数 ──────────────────────────────────
CONTEXT_LENGTHS = [2000, 4000, 8000, 16000, 32000]  # 字符数（新增 16K、32K）
DEPTH_PERCENTAGES = [0, 10, 25, 50, 75, 90, 100]    # needle 位置（%）
SAMPLES_PER_CONFIG = 3                               # 每个配置的样本数（从 1 扩大到 3）
# ────────────────────────────────────────────

total = len(CONTEXT_LENGTHS) * len(DEPTH_PERCENTAGES) * SAMPLES_PER_CONFIG
print(f'预计生成 {len(CONTEXT_LENGTHS)} × {len(DEPTH_PERCENTAGES)} × {SAMPLES_PER_CONFIG} = {total} 条样本')

# 粗略费用估算（以 DeepSeek 为主）
avg_input_chars = sum(CONTEXT_LENGTHS) / len(CONTEXT_LENGTHS)
avg_tokens = avg_input_chars / 1.5  # 中文约 1.5 字符/token
cost_deepseek = total * avg_tokens / 1e6 * 1  # DeepSeek ≈ ¥1/M tokens
print(f'\n💰 费用估算（仅 DeepSeek）：约 ¥{cost_deepseek:.2f}')
print(f'💰 三个模型合计估算：约 ¥{cost_deepseek * 5:.1f}（Kimi 较贵）')

预计生成 5 × 7 × 1 = 35 条样本

💰 费用估算（仅 DeepSeek）：约 ¥0.29


In [7]:
if doc_files:
    documents = [load_document(str(f)) for f in doc_files]
    
    samples = build_niah_samples(
        documents=documents,
        context_lengths=CONTEXT_LENGTHS,
        depth_percentages=DEPTH_PERCENTAGES,
        samples_per_config=SAMPLES_PER_CONFIG,
    )
    
    out_path = '../data/processed/niah_dataset.jsonl'
    save_jsonl(samples, out_path)
else:
    print('⚠️  请先放入文档再运行此单元格')

✅ 保存 35 条样本 → ../data/processed/niah_dataset.jsonl


## Step 5：验证数据集

In [8]:
dataset_path = '../data/processed/niah_dataset.jsonl'

if Path(dataset_path).exists():
    samples = load_jsonl(dataset_path)
    df = pd.DataFrame([
        {
            'context_length': s['context_length'],
            'depth_pct': s['depth_pct'],
            'actual_chars': len(s['context']),
            'answer': s['answer'],
        }
        for s in samples
    ])
    
    print(f'样本总数: {len(df)}')
    print('\n上下文长度分布:')
    print(df.groupby('context_length')['actual_chars'].agg(['count', 'mean', 'min', 'max']).to_string())
    
    print('\n随机抽查一条样本：')
    s = samples[len(samples)//2]
    print(f'  context_length: {s["context_length"]}')
    print(f'  depth_pct:      {s["depth_pct"]}%')
    print(f'  actual_chars:   {len(s["context"])}')
    print(f'  needle:         {s["needle"]}')
    print(f'  question:       {s["question"]}')
    print(f'  answer:         {s["answer"]}')

样本总数: 35

上下文长度分布:
                count          mean   min    max
context_length                                  
2000                7   1882.000000  1868   1894
4000                7   3877.857143  3861   3893
8000                7   7555.857143  5590   7902
16000               7  12045.000000  5588  15902
32000               7  16293.428571   598  31858

随机抽查一条样本：
  context_length: 8000
  depth_pct:      50%
  actual_chars:   7888
  needle:         内部项目代号为“暗星计划”，正式启动日期为 2024 年 11 月 8 日。
  question:       内部项目代号是什么，启动日期是哪天？
  answer:         暗星计划，2024年11月8日


## ✅ 数据准备完成

下一步：打开 `02_eval_runner.ipynb` 开始评测